In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# Uninstall all conflicting packages
!pip uninstall -y numpy scikit-learn sentence-transformers transformers torch torchvision faiss-cpu tokenizers huggingface-hub -y

# Install compatible versions (torch first, then everything else)
!pip install -q torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu118

!pip install -q "numpy>=1.24.0,<2.0.0" \
              "scikit-learn>=1.3.0,<1.6.0" \
              "transformers==4.44.2" \
              "sentence-transformers>=3.0.0" \
              "tokenizers>=0.19.0" \
              "huggingface-hub>=0.24.0" \
              faiss-cpu spacy datasets

!python -m spacy download en_core_web_sm

print("✅ Installed. Please restart runtime now.")

In [1]:
from sentence_transformers import SentenceTransformer



embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

print("✅ Embedder loaded successfully.")

2025-10-31 16:55:23.668195: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1761929723.870577     217 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1761929723.928526     217 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedder loaded successfully.


In [2]:
# ================================

# STEP 1: Imports & Patch

# ================================

import os

os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

os.environ["TRANSFORMERS_OFFLINE"] = "0"

import warnings
warnings.filterwarnings("ignore")
import spacy
import networkx as nx
import faiss
from datasets import load_dataset
import time
import psutil
import torch
import numpy as np



nlp = spacy.load("en_core_web_sm")

print("✅ Libraries loaded.")

✅ Libraries loaded.


In [27]:
def load_targeted_wikipedia_articles(keywords=None, max_articles=50000):

    """

    Load Wikipedia articles specifically about India's economy.

    """

    if keywords is None:

        keywords = ["India"]

    

    print(f"Loading articles containing: {keywords}")

    dataset = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)

    

    articles = []

    titles = []

    urls = []

    

    count = 0

    for item in dataset:

        if count >= max_articles * 10:  # Search more to find relevant ones

            break

        count += 1

        

        title = item["title"]

        text = item["text"]

        

        # MORE STRICT FILTERING: India must be in title OR prominently in text

        india_in_title = "india" in title.lower()

        india_prominent = text.lower().count("india") > 5  # Mentioned multiple times

        

        # Also check for Indian economy specific terms

        economy_terms = ["indian economy", "india's gdp","indian gdp","GDP" ,"Gross Domestic Product" ,"economy", "india's economy", 

                        "economy of india", "gdp of india"]

        has_economy_context = any(term in text.lower()[:50000] for term in economy_terms)

        

        if (india_in_title or india_prominent) and has_economy_context:

            articles.append(text)

            titles.append(title)

            urls.append(item["url"])

            

            if len(articles) >= max_articles:

                break

    

    # If still not enough, add synthetic India-specific data

    if len(articles) < 50:

        print(f"⚠️ Only found {len(articles)} articles. Adding fallback India data.")

        fallback = [

            

        ]

        

        # Replicate fallback data

        for _ in range(10):

            articles.extend(fallback)

            titles.extend(["India Economy and GDP"] * len(fallback))

            urls.extend(["https://example.com/india-economy"] * len(fallback))

    

    print(f"✅ Loaded {len(articles)} India-focused articles.")

    return articles[:max_articles], titles[:max_articles], urls[:max_articles]





# Load India-specific articles

articles, titles, urls = load_targeted_wikipedia_articles(

    keywords=["India"],

    max_articles=50000

)

Loading articles containing: ['India']


Resolving data files:   0%|          | 0/41 [00:00<?, ?it/s]

✅ Loaded 697 India-focused articles.


In [4]:
# ================================

# STEP 3: Build Local Knowledge Graph (Optional)

# ================================

def build_kg_from_articles(texts, titles, max_articles=200):

    G = nx.Graph()

    for idx, (title, text) in enumerate(zip(titles[:max_articles], texts[:max_articles])):

        doc = nlp((title + ". " + text)[:800])

        entities = [ent.text.strip() for ent in doc.ents if ent.label_ in ("PERSON", "ORG", "GPE", "LOC", "MONEY")]

        for i, e1 in enumerate(entities):

            for e2 in entities[i+1:]:

                if e1 and e2:

                    G.add_edge(e1, e2)

    return G



kg = build_kg_from_articles(articles, titles)

print(f"✅ KG built: {len(kg.nodes())} nodes")

✅ KG built: 1454 nodes


In [28]:
texts_for_embedding = [f"{title}: {text[:500]}" for title, text in zip(titles, articles)]
article_embeddings = embedder.encode(
    texts_for_embedding,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

dimension = article_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(article_embeddings)
print(f"✅ FAISS index built: {index.ntotal} docs")

Batches:   0%|          | 0/22 [00:00<?, ?it/s]

✅ FAISS index built: 697 docs


In [24]:
# ================================

# STEP 5: Load Phi-3-mini

# ================================

from transformers import AutoTokenizer, AutoModelForCausalLM



print("🧠 Loading Phi-3-mini...")



model_name = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(

    model_name,

    device_map="auto",

    torch_dtype="auto",

    trust_remote_code=True,

    attn_implementation="eager"

)

print("✅ Phi-3-mini loaded.")

🧠 Loading Phi-3-mini...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

✅ Phi-3-mini loaded.


In [30]:
# ================================
# STEP 6: RAG Pipeline with Chat Format
# ================================
def rag_pipeline(query, k=3):
    # Retrieve
    query_emb = embedder.encode([query], convert_to_numpy=True)
    _, indices = index.search(query_emb, k)
    retrieved = [articles[i] for i in indices[0]]
    context = " ".join([p[:1200] for p in retrieved])

    # Chat-formatted prompt
    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant. Answer the user's question using ONLY the provided context. "
                "If the context doesn't contain the answer, say 'I don't know based on the provided context.'"
            )
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {query}"
        }
    ]

    input_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=3500,
        truncation=True
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "<|assistant|>" in answer:
        answer = answer.split("<|assistant|>")[-1].strip()
    return answer

In [31]:
# ================================

# STEP 7: Run Queries

# ================================

queries = [

    "What is India's GDP?",

    "What was India's GDP growth rate in 2023?",

    "Which sector contributes most to India's economy?",

    "What is the GDP of the United States?"

]



for q in queries:

    print(f"\n❓ {q}")

    print(f"✅ Answer: {rag_pipeline(q, k=3)}")


❓ What is India's GDP?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Answer: You are a helpful assistant. Answer the user's question using ONLY the provided context. If the context doesn't contain the answer, say 'I don't know based on the provided context.' Context:
The history of agriculture in India dates back to the Neolithic period. India ranks second worldwide in farm outputs. As per the Indian economic survey 2020 -21, agriculture employed more than 50% of the Indian workforce and contributed 20.2% to the country's GDP.

In 2016, agriculture and allied sectors like animal husbandry, forestry and fisheries accounted for 17.5% of the GDP (gross domestic product) with about 41.49% of the workforce in 2020. India ranks first in the world with highest net cropped area followed by US and China. The economic contribution of agriculture to India's GDP is steadily declining with the country's broad-based economic growth. Still, agriculture is demographically the broadest economic sector and plays a significant role in the overall socio-economic fabric o

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Answer: You are a helpful assistant. Answer the user's question using ONLY the provided context. If the context doesn't contain the answer, say 'I don't know based on the provided context.' Context:
The history of agriculture in India dates back to the Neolithic period. India ranks second worldwide in farm outputs. As per the Indian economic survey 2020 -21, agriculture employed more than 50% of the Indian workforce and contributed 20.2% to the country's GDP.

In 2016, agriculture and allied sectors like animal husbandry, forestry and fisheries accounted for 17.5% of the GDP (gross domestic product) with about 41.49% of the workforce in 2020. India ranks first in the world with highest net cropped area followed by US and China. The economic contribution of agriculture to India's GDP is steadily declining with the country's broad-based economic growth. Still, agriculture is demographically the broadest economic sector and plays a significant role in the overall socio-economic fabric o

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Answer: You are a helpful assistant. Answer the user's question using ONLY the provided context. If the context doesn't contain the answer, say 'I don't know based on the provided context.' Context:
The history of agriculture in India dates back to the Neolithic period. India ranks second worldwide in farm outputs. As per the Indian economic survey 2020 -21, agriculture employed more than 50% of the Indian workforce and contributed 20.2% to the country's GDP.

In 2016, agriculture and allied sectors like animal husbandry, forestry and fisheries accounted for 17.5% of the GDP (gross domestic product) with about 41.49% of the workforce in 2020. India ranks first in the world with highest net cropped area followed by US and China. The economic contribution of agriculture to India's GDP is steadily declining with the country's broad-based economic growth. Still, agriculture is demographically the broadest economic sector and plays a significant role in the overall socio-economic fabric o

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Answer: You are a helpful assistant. Answer the user's question using ONLY the provided context. If the context doesn't contain the answer, say 'I don't know based on the provided context.' Context:
Karnataka is one of the highest economic growth states in India with an expected GSDP (Gross State Domestic Product) growth of 9.5% in the 2021–22 fiscal year. The total expected GSDP of Karnataka in 2022–2023 is about $240 billion. Karnataka recorded the highest growth rates in terms of GDP and per capita GDP in the last decade compared to other states. In 2008–09, the tertiary sector contributed the most to GSDP (US$31.6 billion─55 percent), followed by the secondary sector ($17 billion─29 percent), and the primary sector (US$9.5 billion─16 percent).

With an overall GDP growth of 56.2% and a per capita GDP growth of 43.9% in the last decade, Karnataka surpassed all other states in India, pushing Karnataka's per capita income in Indian Rupee terms to sixth place. Karnataka received US$2